# Lesson 89 · A complete RAG pipeline — giving an LLM a library: chunking, retrieval, prompts, generation

**Learning goals**
- Assemble L88's TF-IDF retriever into a complete RAG pipeline
- Implement chunk_text() for document chunking
- Implement format_rag_prompt() as a prompt template
- Run it end to end: question → retrieval → fill in the context → generate an answer

> **Three steps**: retrieve → augment → generate. The failure mode: if retrieval is wrong, the generation can be as polished as you like and still be nonsense.

← **Previous lesson** · [L88 · TF-IDF retrieval from scratch](L88_tfidf_retrieval.ipynb)

> Last lesson covered **TF-IDF retrieval from scratch**: term frequency × inverse document frequency, vector search in pure NumPy.
> This lesson explores **a complete RAG pipeline**.

## How does ChatGPT know it should go and look something up?

A large language model's knowledge stops at its training data — it can't answer "What does your company's API documentation say?"

**The RAG (Retrieval-Augmented Generation) answer**:

1. First **retrieve** — find the most relevant passages in your document collection
2. Then **augment** — stuff those passages into the prompt
3. Finally **generate** — let the LLM answer using that context

Think of a library: instead of asking the librarian to answer from memory, you have them pull the relevant pages off the shelf first and answer from what those pages say.

```
User question
   ↓
[chunk] long documents → small chunks, for precise matching
   ↓
[retrieve] TF-IDF cosine search → top-k relevant chunks
   ↓
[augment] fill the chunks into a prompt template
   ↓
[generate] the LLM answers from the context + cites its sources
```

**Why is TF-IDF good enough?**
- No GPU, no pretrained embedding model
- Keyword matching ("FFT", "Nyquist") works very well on technical documents
- The whole retrieval stack can run completely offline

In this lesson you implement `chunk_text` (sliding-window chunking) and `format_rag_prompt` (the prompt template), and run a complete RAG pipeline on top of the TF-IDF retriever in `aurora.llm`.

In [ ]:
import numpy as np
from aurora.llm import build_tfidf, cosine_retrieve

In [ ]:
# Knowledge base (audio AI, longer passages)
KNOWLEDGE_BASE = [
    """The Fast Fourier Transform (FFT) is an algorithm that computes the Discrete Fourier Transform
in O(N log N) time. It was developed by Cooley and Tukey in 1965. The FFT works by recursively
splitting the DFT into even and odd sub-problems, known as the Cooley-Tukey radix-2 algorithm.""",

    """The Short-Time Fourier Transform (STFT) applies a windowed FFT to a signal at successive
time steps to produce a spectrogram. It provides a time-frequency representation showing how the
frequency content of a signal changes over time. The window size controls the time-frequency
resolution tradeoff.""",

    """Mel Frequency Cepstral Coefficients (MFCCs) are widely used features in speech recognition.
They are computed by applying a filterbank of triangular filters on the mel scale, taking the
log, and applying a Discrete Cosine Transform. The mel scale approximates the human auditory
system's logarithmic frequency perception.""",

    """CTC (Connectionist Temporal Classification) is a training objective for sequence-to-sequence
models where the alignment between input and output is unknown. It introduces a blank symbol and
sums over all possible alignments using a forward-backward algorithm. CTC is used in speech
recognition models like DeepSpeech.""",

    """Whisper is a speech recognition model from OpenAI trained on 680,000 hours of multilingual
audio. It uses a Transformer encoder-decoder architecture with log-Mel spectrogram inputs.
Whisper can transcribe speech in multiple languages and translate to English.""",

    """The Nyquist-Shannon sampling theorem states that a signal must be sampled at a rate at least
twice its highest frequency component to be reconstructed without aliasing. For example, CD audio
uses 44,100 Hz to capture frequencies up to 22,050 Hz, covering the full range of human hearing.""",
]
print(f'Knowledge base: {len(KNOWLEDGE_BASE)} passages')

## ✏️ Task 1: document chunking with `chunk_text`

Sending long documents straight into retrieval lowers precision (one chunk covers too many different topics). **Sliding-window chunking** cuts a document into small overlapping pieces; the overlap makes sure important information doesn't get cut in half.

**Signature**:
```python
def chunk_text(text: str, max_words: int = 80, overlap: int = 10) -> list[str]:
```

**Three-step route**:

| Step | What to do | Code hint |
|---|---|---|
| 1 | Split into words | `words = text.split()` |
| 2 | Slide the window | `step = max_words - overlap`; start at 0 and advance by step each time |
| 3 | Join back into strings | store `' '.join(words[i:i+max_words])` in a list |

**Acceptance criteria**:
- Short text (≤ max_words words) → returns `[text]` (a single chunk)
- A 200-word text with max_words=80, overlap=10 → several chunks, each ≤ 80 words
- Neighboring chunks share `overlap` words

In [ ]:
def chunk_text(text: str, max_words: int = 80, overlap: int = 10) -> list:
    """Cut a long text into overlapping chunks of words."""
    words = text.split()
    if len(words) <= max_words:
        return [text]
    # ✏️ TODO: sliding-window chunking
    raise NotImplementedError("TODO")

# Test
try:
    long_text = " ".join(["word"] * 200)
    chunks = chunk_text(long_text, max_words=80, overlap=10)
    print(f"200-word text → {len(chunks)} chunks (max_words=80, overlap=10)")
except (NotImplementedError, TypeError):
    print("⬜ chunk_text not implemented")

# Chunk the knowledge base (with a fallback: use whole passages if not implemented)
all_chunks = []
for doc in KNOWLEDGE_BASE:
    try:
        all_chunks.extend(chunk_text(doc, max_words=60))
    except (NotImplementedError, TypeError):
        all_chunks.append(doc)  # fallback: keep the whole passage if not implemented
print(f"Knowledge base after chunking: {len(all_chunks)} chunks")


In [ ]:
try:
    long_text = ' '.join(['word'] * 200)
    chunks = chunk_text(long_text, max_words=80, overlap=10)
    assert chunks is not None, "⬜ chunk_text not implemented"
    assert isinstance(chunks, list), f"should return a list, got {type(chunks)}"
    assert len(chunks) > 1, f"a 200-word text should be cut into several chunks, got {len(chunks)} chunk(s)"
    for i, chunk in enumerate(chunks):
        assert len(chunk.split()) <= 80, f"chunk {i} has {len(chunk.split())} words, more than max_words=80"
    short_chunks = chunk_text("hello world", max_words=80)
    assert len(short_chunks) == 1, "a short text should stay a single chunk"
    print(f'✅ chunk_text check passed: 200 words → {len(chunks)} chunks, each ≤ 80 words')
except (NotImplementedError, TypeError):
    print('⬜ chunk_text not implemented')

## ✏️ Task 2: the RAG prompt template `format_rag_prompt`

Once the relevant passages are retrieved, they have to be **organized into a prompt** for the LLM. The prompt format directly affects generation quality — numbering the passages lets the LLM cite its sources, and a fixed format tells the LLM where to start writing its answer.

**Signature**:
```python
def format_rag_prompt(query: str, retrieved_docs: list) -> str:
    # retrieved_docs: [(doc_str, score), ...]
```

**Expected output format**:
```
Context:
[1] <first retrieved passage>
[2] <second retrieved passage>

Question: <user question>

Answer:
```

**Two-step route**:

| Step | What to do | Code hint |
|---|---|---|
| 1 | Build the Context paragraphs | `f"[{i+1}] {doc}"` for each `(doc, score)` in retrieved_docs |
| 2 | Join the full prompt | `"Context:\n" + paragraphs + "\n\nQuestion: " + query + "\n\nAnswer:"` |

**Acceptance criteria**:
- the prompt contains the query string
- the prompt contains the content of every retrieved document
- it has `[1]`, `[2]` number markers and ends with `Answer:`

In [ ]:
def format_rag_prompt(query: str, retrieved_docs: list) -> str:
    """Fill the retrieved passages into the prompt template.

    Expected output format:
    Context:
    [1] <first retrieved passage>
    [2] <second retrieved passage>

    Question: <query>

    Answer:
    """
    # ✏️ TODO: build a prompt that includes the context
    raise NotImplementedError("TODO")


In [ ]:
try:
    test_docs = [("FFT is an algorithm that computes DFT.", 0.85),
                 ("STFT applies windowed FFT to a signal.", 0.72)]
    prompt = format_rag_prompt("What is FFT?", test_docs)
    assert prompt is not None, "⬜ format_rag_prompt not implemented"
    assert isinstance(prompt, str), f"should return a string, got {type(prompt)}"
    assert "What is FFT?" in prompt, "the prompt should contain the query"
    assert "FFT is an algorithm" in prompt, "the prompt should contain the retrieved document content"
    assert len(prompt) > 50, f"prompt too short ({len(prompt)} characters)"
    print(f'✅ format_rag_prompt check passed (prompt length {len(prompt)} characters)')
except (NotImplementedError, TypeError, NameError):
    print('⬜ format_rag_prompt not implemented')

In [ ]:
# Build the TF-IDF index
try:
    matrix, vocab = build_tfidf(all_chunks)
    print(f'Index: {matrix.shape[0]} chunks, vocabulary size {matrix.shape[1]}')
except (NotImplementedError, TypeError):
    matrix, vocab = None, []
    print('⚠️ build_tfidf not implemented yet; skipping the RAG pipeline demo. Re-run this cell after finishing the TODO.')

# RAG pipeline demo
if matrix is not None:
    QUERIES = [
        'How does the FFT algorithm work?',
        'What are MFCCs used for?',
        'How does CTC handle alignment in speech recognition?',
    ]

    for query in QUERIES:
        print(f"\n{'='*55}")
        print(f'Question: {query}')
        print('-'*55)
        results = cosine_retrieve(query, matrix, vocab, all_chunks, top_k=2)
        # Print the retrieval results
        for doc, score in results:
            print(f'  [{score:.3f}] {doc[:80]}...')
        # Augment + generate the prompt (skipped gracefully if format_rag_prompt isn't implemented)
        try:
            prompt = format_rag_prompt(query, results)
            print()
            print('Generated prompt (first 200 characters):')
            print(prompt[:200] + '...')
        except (NotImplementedError, TypeError):
            print('  ⬜ format_rag_prompt not implemented — skipping the prompt display')
else:
    print('⚠️ build_tfidf not implemented yet; skipping the RAG pipeline demo.')

In [ ]:
# End-to-end RAG check
if matrix is None:
    print('⬜ build_tfidf not implemented yet; skipping the end-to-end check')
else:
    try:
        test_query = "How does the FFT algorithm work?"
        results = cosine_retrieve(test_query, matrix, vocab, all_chunks, top_k=3)
        assert results is not None, "⬜ cosine_retrieve not implemented"
        assert isinstance(results, list), f"should return a list, got {type(results)}"
        assert len(results) <= 3, f"top_k=3 but {len(results)} results came back"
        for doc, score in results:
            assert isinstance(doc, str), f"a document should be a string, got {type(doc)}"
            assert isinstance(score, float), f"a score should be a float, got {type(score)}"
            assert 0.0 <= score <= 1.0, f"cosine similarity {score} is outside [0, 1]"
        prompt = format_rag_prompt(test_query, results)
        assert isinstance(prompt, str) and len(prompt) > 0, "the prompt should be a non-empty string"
        print(f'✅ End-to-end RAG check passed: retrieved {len(results)} chunks, prompt length {len(prompt)} characters')
    except (NotImplementedError, TypeError, NameError):
        print('⬜ not implemented')

In [ ]:
# Optional: plug in a real LLM to generate an answer
try:
    from transformers import pipeline
    print('(if transformers + a model are installed, you can generate a real answer here)')
    # generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B')
    # answer = generator(prompt, max_new_tokens=100)[0]['generated_text']
except ImportError:
    print('transformers not installed — the RAG retrieval part (no GPU needed) ran in full ✅')

## Wrap-up

| Component | Technique | Dependencies |
|---|---|---|
| Document chunking | sliding window | pure Python |
| Index building | TF-IDF | aurora.llm + NumPy |
| Retrieval | cosine similarity k-NN | aurora.llm + NumPy |
| Prompt template | string formatting | pure Python |
| Generation | LLM (optional) | transformers (optional) |

**Key insight**: retrieval quality decides generation quality. Good retrieval > a big model.

Next up, L90: conversational RAG — conversation memory, source attribution and a podcast Q&A pipeline.

## ✏️ Closed-book derivation check cell — the RAG data-flow diagram

**Rule: close every cell above and work the following out from memory.**

**Problem**: draw the data-flow diagram of the complete RAG pipeline, labeling the input/output type of every step:

| Step | Operation | Input | Output |
|------|------|------|------|
| 1 | Chunking | raw document (str) | List[str] (list of chunks) |
| 2 | Building the TF-IDF index | List[str] | ___ |
| 3 | Vectorizing the query | user question (str) | ___ |
| 4 | Cosine retrieval | ___ | top-k document chunks |
| 5 | Building the prompt | question + top-k chunks | ___ |
| 6 | LLM generation | ___ | answer (str) |

And explain: why does TF-IDF retrieval not need an embedding model?

(Fill in the table here...)

In [ ]:
# Check: RAG retrieval smoke test (no LLM step, only tests the aurora.llm retrieval API)
from aurora.llm import build_tfidf, cosine_retrieve

docs = [
    "Aurora is an audio AI system built from scratch",
    "FFT stands for Fast Fourier Transform, with time complexity O(N log N)",
    "MFCC is the most widely used feature in speech recognition and includes a DCT decorrelation step",
    "Whisper was released by OpenAI and uses the Transformer architecture",
    "A KV cache cuts the total projection cost of inference from O(T²) to O(T)",
]
sm_matrix, sm_vocab = build_tfidf(docs)
query = "What is the time complexity of the FFT"
results = cosine_retrieve(query, sm_matrix, sm_vocab, docs, top_k=2)
assert len(results) == 2, f"expected 2 results, got {len(results)}"
assert any("FFT" in doc for doc, _ in results), "the top-2 results should include the FFT document"
print("Retrieval results:")
for i, (doc, score) in enumerate(results, 1):
    print(f"  {i}. [{score:.3f}] {doc[:50]}...")
print("✅ RAG retrieval smoke test passed")

In [ ]:
# ✏️ Self-review for this lesson
l89_review = {
    "rag_pipeline_map":          None,  # can draw the four-step RAG data flow (chunk/retrieve/augment/generate)? True/False
    "chunk_text_impl":           None,  # chunk_text() sliding window is correct and the checker passes? True/False
    "format_prompt_impl":        None,  # format_rag_prompt() has the right format (numbering + Answer:)? True/False
    "end_to_end_passed":         None,  # the end-to-end RAG smoke test passes (cosine_retrieve + prompt)? True/False
    "whiteboard_passed":         None,  # cleared the whiteboard challenge of filling in the RAG data-flow table? True/False
}

unfilled = [k for k, v in l89_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l89_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L89 all clear! On to L90: conversational RAG')

---

→ **Next lesson** · [L90 · Conversational RAG and a tool-using agent (ReAct)](L90_agent.ipynb)

> Next lesson covers **conversational RAG and a tool-using agent (ReAct)**: conversation memory, source attribution and a podcast Q&A pipeline, and on top of that a ReAct agent written from scratch that runs the Thought → Action → Observation loop.